<a href="https://colab.research.google.com/github/Decoding-Data-Science/airesidency/blob/main/mc14_langchain_pdf_rag_gradio_11jul2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Simple PDF RAG Chatbot (LangChain + OpenAI + Chroma + Gradio)

Upload PDFs, ask questions about them, get answers in a chat window. That's it.

**Before you start:** open the key icon (Secrets) on the left sidebar in Colab and add:
- `OPENAI_API_KEY` -> your OpenAI key
- Turn on notebook access for it

**How to run:** Run the cells top to bottom, in order. After the install cell, restart the runtime once (instructions are in that cell), then keep running down.


## Step 1 - Install packages (run once, then restart runtime)

In [ ]:
!pip -q uninstall -y gradio gradio_client
!pip -q install -U --no-cache-dir langchain langchain-community langchain-openai langchain-chroma chromadb pypdf gradio


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 124.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.9/136.9 kB 80.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 207.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 318.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 158.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 351.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.0/31.0 MB 284.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 281.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 342.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 116.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 329.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.3/558.3 kB 222.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

**Now click Runtime -> Restart session.** Then continue with Step 2 below (you don't need to run Step 1 again).

## Step 2 - Imports

In [ ]:
import os
import shutil
import uuid
from pathlib import Path
from operator import itemgetter

import gradio as gr
import chromadb

from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda

print("Gradio version:", gr.__version__)


## Step 3 - Load your OpenAI key from Colab Secrets

In [ ]:
from google.colab import userdata
import os
os.environ["OPENAI_API_KEY"] = userdata.get("openai")
print("Key loaded:", bool(os.environ.get("OPENAI_API_KEY")))


Key loaded: True


## Step 4 - Set up the model, embeddings, and prompt

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
import chromadb

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are a helpful assistant answering questions about the user's uploaded PDFs.
Use only the retrieved context below to answer. If the answer isn't in the context, say so clearly.""",
    ),
    (
        "human",
        """Conversation so far:
{chat_history}

Question:
{input}

Retrieved context from the documents:
{context}""",
    ),
])


def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)


# In-memory vector store client -- no files written to disk, so there's
# nothing that can get locked or corrupted between runs.
chroma_client = chromadb.EphemeralClient()

app_state = {"retriever": None, "ready": False}

print("Setup complete.")

Setup complete.


## Step 5 - Gradio app: upload PDFs, index them, and chat

In [ ]:
from pathlib import Path
import shutil
import uuid

import gradio as gr
from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_chroma import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda
from operator import itemgetter

DATA_DIR = Path("/content/data")


def process_pdfs(files_list):
    if not files_list:
        return "Please upload at least one PDF first.", gr.update(interactive=False)

    if DATA_DIR.exists():
        shutil.rmtree(DATA_DIR)
    DATA_DIR.mkdir(parents=True)

    for f in files_list:
        src_path = Path(f.name if hasattr(f, "name") else f)
        shutil.copy(src_path, DATA_DIR / src_path.name)

    loader = PyPDFDirectoryLoader(str(DATA_DIR))
    docs = loader.load()
    if not docs:
        return "No readable text found in the uploaded PDF(s).", gr.update(interactive=False)

    chunks = text_splitter.split_documents(docs)

    # A fresh, uniquely-named collection every time -- avoids any leftover
    # state or conflicts from a previous upload.
    collection_name = f"rag_{uuid.uuid4().hex[:8]}"
    vectorstore = Chroma(collection_name=collection_name, embedding_function=embeddings, client=chroma_client)
    vectorstore.add_documents(chunks)

    app_state["retriever"] = vectorstore.as_retriever(search_kwargs={"k": 3})
    app_state["ready"] = True

    return f"Indexed {len(docs)} page(s) / {len(chunks)} chunk(s). You can start chatting now.", gr.update(interactive=True)


def format_history(chat_history):
    if not chat_history:
        return "(no earlier messages)"
    lines = []
    for turn in chat_history:
        role = "User" if turn["role"] == "user" else "Assistant"
        lines.append(f"{role}: {turn['content']}")
    return "\n".join(lines)


def respond(message, chat_history):
    if not message or not message.strip():
        return "", chat_history

    if not app_state["ready"]:
        answer = "Please upload and process PDFs first (top of the page)."
    else:
        retriever = app_state["retriever"]
        chain = (
            {
                "input": itemgetter("input"),
                "chat_history": itemgetter("chat_history"),
                "context": itemgetter("input") | retriever | RunnableLambda(format_docs),
            }
            | prompt
            | llm
            | StrOutputParser()
        )
        answer = chain.invoke({"input": message, "chat_history": format_history(chat_history)})

    chat_history = chat_history + [
        {"role": "user", "content": message},
        {"role": "assistant", "content": answer},
    ]
    return "", chat_history


try:
    demo.close()
except NameError:
    pass

with gr.Blocks(title="PDF RAG Chatbot") as demo:
    gr.Markdown("## Ask questions about your PDFs")

    with gr.Row():
        file_upload = gr.File(file_count="multiple", file_types=[".pdf"], label="Upload PDF(s)")
        process_btn = gr.Button("Process Documents", variant="primary")

    status_box = gr.Markdown("Upload PDFs and click **Process Documents** to begin.")
    chatbot = gr.Chatbot(height=450, label="Chat")
    msg_box = gr.Textbox(placeholder="Ask a question about your uploaded PDFs...", label="Your question", interactive=False)

    process_btn.click(fn=process_pdfs, inputs=[file_upload], outputs=[status_box, msg_box])
    msg_box.submit(fn=respond, inputs=[msg_box, chatbot], outputs=[msg_box, chatbot])

demo.launch(share=True, debug=True)

/tmp/ipykernel_1525/2940329896.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFDirectoryLoader


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://d287bc26a9a19274f3.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
